# Kaggle Demo Launcher

Launcher cepat untuk memastikan `cnn.ipynb` dan `rnn_lstm.ipynb` bisa berjalan end-to-end di Kaggle tanpa menunggu eksperimen penuh. Mode ini memakai subset kecil, 1 epoch, 1 variasi RNN/LSTM, dan menyimpan hasil ke `/kaggle/working/outputs_demo`.

## 0. Configuration

Ubah cell ini saja kalau nama dataset Kaggle berbeda atau mau run sebagian pipeline.

In [ ]:
from pathlib import Path

RUN_CNN_NOTEBOOK = True
RUN_RNN_LSTM_NOTEBOOK = True

# Mode tersedia:
# - "demo": cek cepat end-to-end dengan subset kecil, 1 epoch, 1 variasi.
# - "full": eksperimen final sesuai kebutuhan laporan.
RUN_MODE = "demo"

CODE_DATASET_SLUG = ""
INTEL_DATASET_SLUG = ""
FLICKR8K_DATASET_SLUG = ""

WORK_ROOT = Path("/kaggle/working")
REPO_WORK = WORK_ROOT / "SpreiElsa"
CNN_OUTPUT_DIR = WORK_ROOT / "outputs_demo" / "cnn"
RNN_OUTPUT_DIR = WORK_ROOT / "outputs_demo" / "rnn_lstm"
EXECUTED_DIR = WORK_ROOT / "executed_notebooks_demo"

DEMO_CNN_EPOCHS = 1
DEMO_RNN_EPOCHS = 1
DEMO_RNN_VARIATIONS = [{"n_layers": 1, "hidden_size": 128}]
DEMO_N_EVAL = 5
DEMO_INTEL_TRAIN_PER_CLASS = 12
DEMO_INTEL_VAL_PER_CLASS = 4
DEMO_INTEL_TEST_PER_CLASS = 4
DEMO_FLICKR_TRAIN_IMAGES = 24
DEMO_FLICKR_VAL_IMAGES = 6
DEMO_FLICKR_TEST_IMAGES = 6

FULL_CNN_EPOCHS = 10
FULL_RNN_EPOCHS = 5
FULL_N_EVAL = 50

# Backward-compatible aliases used by the patching cell.
SMOKE_CNN_EPOCHS = DEMO_CNN_EPOCHS
SMOKE_RNN_EPOCHS = DEMO_RNN_EPOCHS
SMOKE_RNN_VARIATIONS = DEMO_RNN_VARIATIONS
SMOKE_N_EVAL = DEMO_N_EVAL


## 1. Check GPU

In [ ]:
import os
import sys
import shutil
import subprocess
import json
import time

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Locate Kaggle Inputs

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
if not INPUT_ROOT.exists():
    raise RuntimeError("Notebook ini harus dijalankan di Kaggle: /kaggle/input tidak ditemukan.")

print("Available input roots:")
for path in sorted(INPUT_ROOT.iterdir()):
    print(" -", path)

In [ ]:
def path_from_slug(slug: str):
    if not slug:
        return None
    p = INPUT_ROOT / slug
    return p if p.exists() else None


def find_repo_root() -> Path:
    explicit = path_from_slug(CODE_DATASET_SLUG)
    candidates = []
    if explicit:
        candidates.extend([explicit, *explicit.glob("**/*")])
    else:
        candidates.extend(INPUT_ROOT.glob("**/*"))
        candidates.insert(0, INPUT_ROOT)
    for p in candidates:
        if p.is_dir() and (p / "src").exists() and (p / "src" / "notebook").exists():
            return p
    raise FileNotFoundError("Repo tidak ditemukan. Upload repo sebagai Kaggle Dataset atau isi CODE_DATASET_SLUG.")


def find_file(name_options, preferred_slug=""):
    roots = []
    explicit = path_from_slug(preferred_slug)
    if explicit:
        roots.append(explicit)
    roots.append(INPUT_ROOT)
    for root in roots:
        for name in name_options:
            matches = sorted(root.glob(f"**/{name}"))
            if matches:
                return matches[0]
    return None


def find_dir(name_options, preferred_slug=""):
    roots = []
    explicit = path_from_slug(preferred_slug)
    if explicit:
        roots.append(explicit)
    roots.append(INPUT_ROOT)
    for root in roots:
        for name in name_options:
            matches = sorted(p for p in root.glob(f"**/{name}") if p.is_dir())
            if matches:
                return matches[0]
    return None

repo_input = find_repo_root()
intel_root = path_from_slug(INTEL_DATASET_SLUG) or find_dir(["intel-image-classification", "seg_train"], INTEL_DATASET_SLUG)

def normalize_intel_root(path):
    if path is None:
        return None
    candidates = [path, *path.parents]
    for candidate in candidates:
        if all((candidate / split).is_dir() for split in ("train", "val", "test")):
            return candidate
        if (candidate / "seg_train").is_dir() and (candidate / "seg_test").is_dir():
            return candidate
    return path

intel_root = normalize_intel_root(intel_root)

flickr_root = path_from_slug(FLICKR8K_DATASET_SLUG)
flickr_images = find_dir(["Flicker8k_Dataset", "Flickr8k_Dataset", "Images"], FLICKR8K_DATASET_SLUG)
flickr_captions = find_file(["captions.txt", "Flickr8k.token.txt"], FLICKR8K_DATASET_SLUG)
flickr_train = find_file(["Flickr_8k.trainImages.txt", "Flickr8k.trainImages.txt"], FLICKR8K_DATASET_SLUG)
flickr_val = find_file(["Flickr_8k.devImages.txt", "Flickr8k.devImages.txt"], FLICKR8K_DATASET_SLUG)
flickr_test = find_file(["Flickr_8k.testImages.txt", "Flickr8k.testImages.txt"], FLICKR8K_DATASET_SLUG)

print("repo_input:", repo_input)
print("intel_root:", intel_root)
print("flickr_images:", flickr_images)
print("flickr_captions:", flickr_captions)
print("flickr_train:", flickr_train)
print("flickr_val:", flickr_val)
print("flickr_test:", flickr_test)

## 3. Copy Repo to Writable Working Directory

In [ ]:
if REPO_WORK.exists():
    shutil.rmtree(REPO_WORK)
shutil.copytree(repo_input, REPO_WORK)
os.chdir(REPO_WORK)

print("Working repo:", REPO_WORK)
print("Notebook files:")
for p in sorted((REPO_WORK / "src" / "notebook").glob("*.ipynb")):
    print(" -", p.relative_to(REPO_WORK))

## 4. Install Dependencies

In [ ]:
req = REPO_WORK / "requirements-dev.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
else:
    print("requirements-dev.txt tidak ditemukan; skip.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pillow", "matplotlib", "nltk"], check=True)

## 5. Set Environment Variables

In [ ]:
RUN_MODE = RUN_MODE.lower().strip()
if RUN_MODE == "smoke":
    RUN_MODE = "demo"
if RUN_MODE not in {"demo", "full"}:
    raise ValueError("RUN_MODE harus 'demo' atau 'full'")
DEMO_MODE = RUN_MODE == "demo"

if str(REPO_WORK) not in sys.path:
    sys.path.insert(0, str(REPO_WORK))

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _caption_image_names(path: Path):
    names = []
    if not path or not path.exists():
        return names
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line or line.lower().startswith("image,caption"):
                continue
            left = line.split("\t", 1)[0] if "\t" in line else line.split(",", 1)[0]
            name = left.split("#", 1)[0].strip()
            if name:
                names.append(name)
    return sorted(set(names))


def _write_split(path: Path, names):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(names) + "\n", encoding="utf-8")


def _image_map(image_dir: Path):
    if image_dir is None or not image_dir.exists():
        return {}
    return {p.name: p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}


if DEMO_MODE:
    print("RUN_MODE=demo: membuat subset kecil agar notebook cepat mengecek semua cell.")

    if intel_root is not None:
        from src.dataset.prepare_intel_dataset import prepare_intel_dataset

        demo_intel_root = WORK_ROOT / "intel_demo"
        if demo_intel_root.exists():
            shutil.rmtree(demo_intel_root)
        prepare_intel_dataset(
            source_root=Path(intel_root),
            output_root=demo_intel_root,
            val_fraction=0.15,
            overwrite=True,
            max_train_per_class=DEMO_INTEL_TRAIN_PER_CLASS,
            max_val_per_class=DEMO_INTEL_VAL_PER_CLASS,
            max_test_per_class=DEMO_INTEL_TEST_PER_CLASS,
        )
        intel_root = demo_intel_root
        print("Demo Intel dataset:", demo_intel_root)

    if flickr_images is not None and flickr_captions is not None:
        image_lookup = _image_map(Path(flickr_images))
        caption_names = [name for name in _caption_image_names(Path(flickr_captions)) if name in image_lookup]
        if not caption_names:
            caption_names = sorted(image_lookup)
        need = DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES + DEMO_FLICKR_TEST_IMAGES
        selected = caption_names[:need]
        if len(selected) < 3:
            raise RuntimeError("Demo Flickr8k gagal: gambar/caption yang cocok terlalu sedikit.")

        demo_flickr_images = WORK_ROOT / "flickr8k_demo_images"
        if demo_flickr_images.exists():
            shutil.rmtree(demo_flickr_images)
        demo_flickr_images.mkdir(parents=True, exist_ok=True)
        for name in selected:
            shutil.copy2(image_lookup[name], demo_flickr_images / name)

        split_dir = WORK_ROOT / "flickr8k_demo_splits"
        train_names = selected[:DEMO_FLICKR_TRAIN_IMAGES]
        val_names = selected[DEMO_FLICKR_TRAIN_IMAGES:DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES]
        test_names = selected[DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES:]
        if not val_names:
            val_names = train_names[-1:]
        if not test_names:
            test_names = val_names[-1:]

        flickr_images = demo_flickr_images
        flickr_train = split_dir / "Flickr_8k.trainImages.txt"
        flickr_val = split_dir / "Flickr_8k.devImages.txt"
        flickr_test = split_dir / "Flickr_8k.testImages.txt"
        _write_split(flickr_train, train_names)
        _write_split(flickr_val, val_names)
        _write_split(flickr_test, test_names)
        print("Demo Flickr images:", len(selected), demo_flickr_images)
        print("Demo Flickr splits:", len(train_names), len(val_names), len(test_names))

# Fallback jika dataset Flickr8k tidak menyediakan split file resmi pada mode full.
if flickr_captions is not None and not all([flickr_train, flickr_val, flickr_test]):
    split_dir = WORK_ROOT / "flickr8k_generated_splits"
    names = _caption_image_names(Path(flickr_captions))
    if not names and flickr_images is not None:
        names = sorted(_image_map(Path(flickr_images)))
    if len(names) < 3:
        raise RuntimeError("Tidak bisa generate split Flickr8k: nama gambar terlalu sedikit atau captions tidak terbaca.")
    train_names = names[: min(6000, max(1, int(0.75 * len(names))))]
    val_start = len(train_names)
    val_end = min(val_start + 1000, val_start + max(1, int(0.125 * len(names))))
    val_names = names[val_start:val_end]
    test_names = names[val_end:]
    if not test_names:
        test_names = val_names[-max(1, len(val_names)//2):]
        val_names = val_names[:-len(test_names)] or test_names
    flickr_train = split_dir / "Flickr_8k.trainImages.txt"
    flickr_val = split_dir / "Flickr_8k.devImages.txt"
    flickr_test = split_dir / "Flickr_8k.testImages.txt"
    _write_split(flickr_train, train_names)
    _write_split(flickr_val, val_names)
    _write_split(flickr_test, test_names)
    print("Generated Flickr8k splits:", len(train_names), len(val_names), len(test_names))

if intel_root is not None:
    os.environ["INTEL_DATA_ROOT"] = str(intel_root)
os.environ["INTEL_PREPARED_ROOT"] = str(WORK_ROOT / ("intel_demo" if DEMO_MODE else "intel_prepared"))
os.environ["CNN_OUTPUT_DIR"] = str(CNN_OUTPUT_DIR)

if flickr_root is not None:
    os.environ["FLICKR8K_ROOT"] = str(flickr_root)
if flickr_images is not None:
    os.environ["FLICKR8K_IMAGES"] = str(flickr_images)
if flickr_captions is not None:
    os.environ["FLICKR8K_CAPTIONS"] = str(flickr_captions)
if flickr_train is not None:
    os.environ["FLICKR8K_TRAIN_SPLIT"] = str(flickr_train)
if flickr_val is not None:
    os.environ["FLICKR8K_VAL_SPLIT"] = str(flickr_val)
if flickr_test is not None:
    os.environ["FLICKR8K_TEST_SPLIT"] = str(flickr_test)
os.environ["WORKING_DIR"] = str(RNN_OUTPUT_DIR)

print("RUN_MODE=", RUN_MODE)
print("INTEL_DATA_ROOT=", os.environ.get("INTEL_DATA_ROOT"))
print("CNN_OUTPUT_DIR=", os.environ.get("CNN_OUTPUT_DIR"))
print("FLICKR8K_IMAGES=", os.environ.get("FLICKR8K_IMAGES"))
print("FLICKR8K_CAPTIONS=", os.environ.get("FLICKR8K_CAPTIONS"))
print("FLICKR8K_TRAIN_SPLIT=", os.environ.get("FLICKR8K_TRAIN_SPLIT"))
print("WORKING_DIR=", os.environ.get("WORKING_DIR"))


## 6. Patch Notebook Flags in Working Copy

In [ ]:
def load_notebook(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_notebook(path: Path, nb: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(nb, handle, indent=1)


def replace_code_cell_containing(nb: dict, needle: str, new_source: str) -> None:
    for cell in nb["cells"]:
        if cell.get("cell_type") == "code" and needle in "".join(cell.get("source", [])):
            cell["source"] = new_source.strip().splitlines(True)
            return
    raise ValueError(f"Cell containing {needle!r} not found")


def patch_cnn_notebook(path: Path) -> Path:
    nb = load_notebook(path)
    full = RUN_MODE == "full"
    source = f'''
repo_root = Path(".").resolve()
is_kaggle = Path("/kaggle/input").exists()

intel_source_root = Path(os.getenv("INTEL_DATA_ROOT", "/kaggle/input/intel-image-classification" if is_kaggle else "src/dataset"))
prepared_root = Path(os.getenv("INTEL_PREPARED_ROOT", "/kaggle/working/intel_prepared" if is_kaggle else "src/dataset"))
cnn_output_dir = Path(os.getenv("CNN_OUTPUT_DIR", "/kaggle/working/outputs/cnn" if is_kaggle else "outputs/cnn"))
cnn_output_dir.mkdir(parents=True, exist_ok=True)

cnn_image_size = 64
cnn_experiment_id = "d1_f16_k3_max"
cnn_non_shared = False
cnn_epochs = {FULL_CNN_EPOCHS if full else SMOKE_CNN_EPOCHS}
cnn_batch_size = 32 if is_kaggle else 16

run_prepare = True
run_train_single = {str(not full)}
run_train_all = {str(full)}
run_non_shared = {str(full)}
'''
    replace_code_cell_containing(nb, "run_prepare =", source)
    out_path = path.with_name("cnn_kaggle_run.ipynb")
    save_notebook(out_path, nb)
    return out_path


def patch_rnn_notebook(path: Path) -> Path:
    nb = load_notebook(path)
    full = RUN_MODE == "full"
    variations = [
        {"n_layers": 1, "hidden_size": 128},
        {"n_layers": 1, "hidden_size": 256},
        {"n_layers": 2, "hidden_size": 128},
        {"n_layers": 2, "hidden_size": 256},
        {"n_layers": 3, "hidden_size": 128},
        {"n_layers": 3, "hidden_size": 256},
    ] if full else SMOKE_RNN_VARIATIONS
    source = f'''
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

RUN_EXTRACT_FEATURES = True
RUN_PREPROCESS = True
RUN_TRAIN_RNN = True
RUN_TRAIN_LSTM = True
RUN_TRAIN_INIT_INJECT = {str(full)}
RUN_FULL_EVAL = True

N_EVAL = {FULL_N_EVAL if full else SMOKE_N_EVAL}
N_BATCH = {FULL_N_EVAL if full else SMOKE_N_EVAL}
BATCH_SIZE = 64
EPOCHS = {FULL_RNN_EPOCHS if full else SMOKE_RNN_EPOCHS}
MAX_SEQ_LEN = 35
EMBED_DIM = 256
FEATURE_DIM = 2048
LEARNING_RATE = 1e-3

VARIATIONS = {json.dumps(variations, indent=4)}
VARIATIONS
'''
    replace_code_cell_containing(nb, "RUN_EXTRACT_FEATURES =", source)
    out_path = path.with_name("rnn_lstm_kaggle_run.ipynb")
    save_notebook(out_path, nb)
    return out_path

cnn_run_nb = patch_cnn_notebook(REPO_WORK / "src" / "notebook" / "cnn.ipynb")
rnn_run_nb = patch_rnn_notebook(REPO_WORK / "src" / "notebook" / "rnn_lstm.ipynb")

print("Patched CNN notebook:", cnn_run_nb)
print("Patched RNN/LSTM notebook:", rnn_run_nb)

## 7. Execute Notebooks

In [ ]:
EXECUTED_DIR.mkdir(parents=True, exist_ok=True)

def execute_notebook(input_path: Path, output_name: str) -> Path:
    output_path = EXECUTED_DIR / output_name
    cmd = [
        "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute", str(input_path),
        "--output", str(output_path),
        "--ExecutePreprocessor.timeout=-1",
        "--ExecutePreprocessor.kernel_name=python3",
    ]
    print("Running:", " ".join(cmd))
    start = time.perf_counter()
    env = os.environ.copy()
    repo_path = str(REPO_WORK)
    env["PYTHONPATH"] = repo_path + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    subprocess.run(cmd, cwd=REPO_WORK, env=env, check=True)
    elapsed = time.perf_counter() - start
    print(f"Done in {elapsed/60:.2f} minutes -> {output_path}")
    return output_path

executed = []
if RUN_CNN_NOTEBOOK:
    executed.append(execute_notebook(cnn_run_nb, "cnn_executed.ipynb"))
if RUN_RNN_LSTM_NOTEBOOK:
    executed.append(execute_notebook(rnn_run_nb, "rnn_lstm_executed.ipynb"))

executed

## 8. Show Output Files

In [ ]:
print("Executed notebooks:")
for p in sorted(EXECUTED_DIR.glob("*.ipynb")):
    print(" -", p)

print("\nCNN outputs:")
if CNN_OUTPUT_DIR.exists():
    for p in sorted(CNN_OUTPUT_DIR.glob("**/*"))[:80]:
        print(" -", p)
else:
    print("CNN output dir belum ada")

print("\nRNN/LSTM outputs:")
if RNN_OUTPUT_DIR.exists():
    for p in sorted(RNN_OUTPUT_DIR.glob("**/*"))[:120]:
        print(" -", p)
else:
    print("RNN output dir belum ada")

## 9. Zip Results for Download

In [ ]:
zip_base = WORK_ROOT / "sprei_elsa_kaggle_demo_results"
if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(str(zip_base), "zip", root_dir=WORK_ROOT, base_dir="outputs_demo")
print("Created:", zip_base.with_suffix(".zip"))
print("Executed notebooks are in:", EXECUTED_DIR)